In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv("..\\data\\processed\\data.csv" , encoding="latin-1")
target = pd.read_csv("..\\data\\processed\\target.csv" , encoding="latin-1")



In [3]:
data.head()

,Year,Month,DayofMonth,DayOfWeek,CRSDepTime,CRSArrTime,CRSElapsedTime,DepTimeBlk,Origin,Dest,Distance,Reporting_Airline,Tail_Number
0,2009,5,28,4,1204,1541,157.0,1200-1259,MKE,MCO,1066.0,FL,N946AT
1,2013,6,29,6,1630,1945,135.0,1600-1659,GJT,DFW,773.0,MQ,N665MQ
2,2010,8,31,2,1305,2035,270.0,1300-1359,LAX,DTW,1979.0,DL,N6705Y
3,2006,1,15,7,1820,2026,126.0,1800-1859,EWR,CLT,529.0,US,N504AU
4,2006,8,7,1,1755,2000,125.0,1700-1759,BOS,CLE,563.0,CO,N27724


In [4]:
data['dep_hour'] = data['CRSDepTime'] // 100

def dep_period(hour):
    if 5 <= hour < 9:   return 'early_morning'
    if 9 <= hour < 12:  return 'morning'
    if 12 <= hour < 17: return 'afternoon'
    if 17 <= hour < 21: return 'evening'
    return 'red_eye'

In [5]:
data['dep_period'] = data['dep_hour'].apply(dep_period)

In [6]:
data['is_weekend'] = data['DayOfWeek'].isin([6, 7]).astype(int)

data['season'] = data['Month'].map({
    12: 'winter', 1: 'winter', 2: 'winter',
    3: 'spring', 4: 'spring', 5: 'spring',
    6: 'summer', 7: 'summer', 8: 'summer',
    9: 'fall',  10: 'fall',  11: 'fall'
})
# Summer-Thanksgiving-XMS & New Year Months
data['is_peak_month'] = data['Month'].isin([6, 7, 8, 11, 12]).astype(int)

In [7]:
data['route'] = data['Origin'] + '_' + data['Dest']

In [8]:
data.head()

,Year,Month,DayofMonth,DayOfWeek,CRSDepTime,CRSArrTime,CRSElapsedTime,DepTimeBlk,Origin,Dest,Distance,Reporting_Airline,Tail_Number,dep_hour,dep_period,is_weekend,season,is_peak_month,route
0,2009,5,28,4,1204,1541,157.0,1200-1259,MKE,MCO,1066.0,FL,N946AT,12,afternoon,0,spring,0,MKE_MCO
1,2013,6,29,6,1630,1945,135.0,1600-1659,GJT,DFW,773.0,MQ,N665MQ,16,afternoon,1,summer,1,GJT_DFW
2,2010,8,31,2,1305,2035,270.0,1300-1359,LAX,DTW,1979.0,DL,N6705Y,13,afternoon,0,summer,1,LAX_DTW
3,2006,1,15,7,1820,2026,126.0,1800-1859,EWR,CLT,529.0,US,N504AU,18,evening,1,winter,0,EWR_CLT
4,2006,8,7,1,1755,2000,125.0,1700-1759,BOS,CLE,563.0,CO,N27724,17,evening,0,summer,1,BOS_CLE


In [9]:
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=42)

train_df = X_train.copy()
train_df['ArrDel15'] = y_train.values

In [10]:
def add_delay_rates(train_df, apply_df, group_col, feature_name):
    rate_map = train_df.groupby(group_col)['ArrDel15'].mean()
    global_mean = train_df['ArrDel15'].mean()  
    apply_df[feature_name] = apply_df[group_col].map(rate_map).fillna(global_mean)
    return apply_df

In [11]:
for df_split in [X_train, X_test]:
    df_split = add_delay_rates(train_df, df_split, 'Reporting_Airline', 'airline_delay_rate')
    df_split = add_delay_rates(train_df, df_split, 'Origin',            'origin_delay_rate')
    df_split = add_delay_rates(train_df, df_split, 'Dest',              'dest_delay_rate')
    df_split = add_delay_rates(train_df, df_split, 'route',             'route_delay_rate')
    df_split = add_delay_rates(train_df, df_split, 'Tail_Number',       'tail_delay_rate')

In [12]:
print("Training features:", X_train.columns.tolist())

Training features: ['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'CRSDepTime', 'CRSArrTime', 'CRSElapsedTime', 'DepTimeBlk', 'Origin', 'Dest', 'Distance', 'Reporting_Airline', 'Tail_Number', 'dep_hour', 'dep_period', 'is_weekend', 'season', 'is_peak_month', 'route', 'airline_delay_rate', 'origin_delay_rate', 'dest_delay_rate', 'route_delay_rate', 'tail_delay_rate']


In [13]:
cols_to_drop = ['Year','route','Tail_Number','CRSDepTime','CRSArrTime','DepTimeBlk','DayofMonth','Month']

X_train.drop(columns=cols_to_drop, inplace=True)
X_test.drop(columns=cols_to_drop, inplace=True)


In [14]:
print("Training features:", X_train.columns.tolist())
print("Shape:", X_train.shape)
print("Shape",X_train.shape)

Training features: ['DayOfWeek', 'CRSElapsedTime', 'Origin', 'Dest', 'Distance', 'Reporting_Airline', 'dep_hour', 'dep_period', 'is_weekend', 'season', 'is_peak_month', 'airline_delay_rate', 'origin_delay_rate', 'dest_delay_rate', 'route_delay_rate', 'tail_delay_rate']
Shape: (1048236, 16)
Shape (1048236, 16)


In [15]:
X_train.head()

,DayOfWeek,CRSElapsedTime,Origin,Dest,Distance,Reporting_Airline,dep_hour,dep_period,is_weekend,season,is_peak_month,airline_delay_rate,origin_delay_rate,dest_delay_rate,route_delay_rate,tail_delay_rate
502661,3,108.0,RSW,ATL,515.0,FL,8,early_morning,0,spring,0,0.205103,0.186818,0.194709,0.175633,0.201835
446685,1,61.0,LAX,IPL,181.0,OO,11,morning,0,summer,1,0.181983,0.191326,0.217391,0.200000,0.193333
1095176,1,110.0,DTW,BOS,632.0,NW,18,evening,0,winter,0,0.211510,0.220821,0.222497,0.256410,0.224242
245163,7,89.0,CLT,MDT,413.0,AA,11,morning,1,winter,0,0.206187,0.215128,0.231707,0.178295,0.157303
1213794,6,318.0,LAX,JFK,2475.0,AA,7,early_morning,1,spring,0,0.206187,0.191326,0.236769,0.219773,0.250000


In [ ]:
X_train.to_csv("..\\data\\processed\\X_train.csv" ,index = False)
X_test.to_csv("..\\data\\processed\\X_test.csv" ,index = False)
y_train.to_csv("..\\data\\processed\\y_train.csv" ,index = False)
y_test.to_csv("..\\data\\processed\\y_test.csv" ,index = False)